# 01 · Landing
Descarga el zip del día desde NOAA y deja el CSV en `landing/csv`, la carpeta que vigila Auto Loader. Es idempotente: si el archivo ya existe no lo repite.

In [0]:
%run ./00_config

In [0]:
from datetime import date, timedelta

dbutils.widgets.text("fecha", "auto")
fecha = dbutils.widgets.get("fecha")

# "auto": siguiente día al último que ya está en plata (simula el feed diario)
if fecha == "auto":
    t = T("silver", "ais_posiciones")
    ult = spark.sql(f"SELECT max(fecha) FROM {t}").first()[0] if spark.catalog.tableExists(t) else None
    fecha = str(ult + timedelta(days=1)) if ult else "2023-06-01"

# Publica la fecha resuelta para las tareas siguientes del job (gold y quality gate)
dbutils.jobs.taskValues.set(key="fecha", value=fecha)
print("fecha:", fecha)

In [0]:
import os, shutil, zipfile, urllib.request

y, m, d = fecha.split("-")
nombre = f"AIS_{y}_{m}_{d}"
url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{y}/{nombre}.zip"

zip_path  = f"{V('raw_zips')}/{nombre}.zip"
csv_dir   = V("csv")
csv_final = f"{csv_dir}/{nombre}.csv"

# 1. Descarga (idempotente)
if not os.path.exists(zip_path):
    print("descargando", url)
    urllib.request.urlretrieve(url, zip_path)
print("zip:", round(os.path.getsize(zip_path) / 1e6, 1), "MB")

# 2. Descompresión a temporal y move a la carpeta vigilada (el CSV aparece completo de una vez)
if not os.path.exists(csv_final):
    tmp_dir = f"{V('raw_zips')}/_tmp_{nombre}"
    os.makedirs(tmp_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        print("contenido del zip:", z.namelist())
        z.extractall(tmp_dir)
    csv_tmp = [os.path.join(r, f) for r, _, fs in os.walk(tmp_dir) for f in fs if f.endswith(".csv")][0]
    shutil.move(csv_tmp, csv_final)
    shutil.rmtree(tmp_dir)

print("csv:", round(os.path.getsize(csv_final) / 1e6, 1), "MB")
for f in dbutils.fs.ls(csv_dir):
    print(f.name, f.size)